# UniLab Demo: 一键回放训练好的策略

这个 notebook 演示如何用 UniLab 的 `demo` 命令，加载一个已经训练好的 checkpoint，
让机器人在仿真环境中执行学到的策略。

**你不需要任何机器人或强化学习背景**——只需要运行下面的 cell 就能看到效果。

## 它做了什么？

1. 从预设的 checkpoint 目录复制模型文件到 demo 运行目录
2. 启动 MuJoCo 物理仿真器
3. 让 Go2 四足机器人用训练好的策略行走
4. 录制一段回放视频

## 环境准备

> 如果你已经完成了 README 中 Quick Demo 的步骤，可以跳过这一节。

**系统要求**：Python >= 3.10, < 3.14

```bash
# 1. 安装 uv（Python 包管理器）
curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. 克隆仓库
git clone <repo-url> && cd UniLab

# 3. 安装依赖
uv sync

# 4. 激活虚拟环境
source .venv/bin/activate

# 5. 验证 CLI 可用
unilab --help
```

**前置条件**：本地需要存在训练产出的 checkpoint 文件（位于 `logs/rsl_rl_ppo/Go2JoystickFlatTerrain/` 下）。
如果你还没有 checkpoint，请先运行 walkthrough notebook 完成一次训练。

> **Colab 不适用**：本项目依赖 MuJoCo 物理仿真器的本地运行环境，不支持 Google Colab。请在本地机器上运行此 notebook。

## Step 1: 确认环境

检查我们是否在正确的项目根目录，以及 `unilab` CLI 是否可用。

In [ ]:
import os
from pathlib import Path

# 确保在项目根目录运行
root = Path(".").resolve()
assert (root / "scripts").is_dir(), f"请在 UniLab 项目根目录运行此 notebook，当前目录: {root}"
assert (root / "conf").is_dir(), "缺少 conf/ 目录，请确认项目完整性"
print(f"项目根目录: {root}")

In [ ]:
!uv run unilab --help

## Step 2: 检查 checkpoint 是否存在

demo 命令需要一个预训练好的模型文件。下面检查它是否在预期位置。

In [ ]:
log_dir = root / "logs" / "rsl_rl_ppo" / "Go2JoystickFlatTerrain"

if log_dir.is_dir():
    runs = sorted(log_dir.iterdir())
    print(f"找到 {len(runs)} 个训练 run:")
    for r in runs:
        ckpts = list(r.glob("model_*.pt"))
        print(f"  {r.name}  ->  {len(ckpts)} 个 checkpoint")
else:
    print(f"未找到日志目录: {log_dir}")
    print("请先完成至少一次训练，或手动放置 checkpoint 文件。")

## Step 3: 运行 Demo

`unilab demo` 会自动完成以下步骤：
- 找到 preset 对应的 checkpoint
- 复制到 demo 专用目录
- 以 play_only 模式启动仿真，录制视频

默认 preset 是 `go2_joystick_mujoco_ppo`（Go2 机器人 + 摇杆控制 + MuJoCo 仿真器 + PPO 算法）。

In [ ]:
!uv run unilab demo

## Step 4: 查看回放视频

如果 demo 成功运行，会在 demo run 目录下生成 `play_video.mp4`。

In [ ]:
from IPython.display import Video

demo_run_dir = log_dir / "demo_go2_joystick_mujoco_ppo_v0"
video_path = demo_run_dir / "play_video.mp4"

if video_path.is_file():
    print(f"视频路径: {video_path}")
    display(Video(str(video_path), embed=True, width=640))
else:
    print(f"未找到视频文件: {video_path}")
    print("可能原因: demo 运行失败，或渲染环境不可用。")

## 下一步

- 想了解训练过程？请查看 `unilab_walkthrough_ppo_go1_joystick_mujoco.ipynb`
- 想用其他算法或机器人？运行 `uv run unilab train --help` 查看所有选项
- 想自定义 demo preset？查看 `src/unilab/demo.py` 中的 `DEMO_PRESETS`